# MedCompress — Kvasir-SEG Polyp Segmentation
**Author: Abhishek Shekhar**

U-Net teacher + lite U-Net student with QAT and KD on Kvasir-SEG polyp segmentation.

> Outputs persist on Drive. Completed experiments are skipped.

In [ ]:
import os, sys, time, random, json, warnings
warnings.filterwarnings("ignore")
import numpy as np

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── GPU check ───────────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")
tf.config.set_visible_devices(gpus[0], 'GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
print(f"TensorFlow {tf.__version__}  GPU: {gpus[0].name}")

# ── Directories (all outputs persist on Drive) ──────────────────────────────
EXP_NAME    = "kvasir_segmentation"   # ← overridden per notebook
DRIVE_BASE  = f"/content/drive/MyDrive/medcompress/kvasir_segmentation"
DATA_DIR    = f"/content/data/kvasir_segmentation"
RESULTS_DIR = f"{DRIVE_BASE}/results"
CKPT_DIR    = f"{DRIVE_BASE}/checkpoints"
MODELS_DIR  = f"{DRIVE_BASE}/models"
TFLITE_DIR  = f"{DRIVE_BASE}/tflite"
for d in [DRIVE_BASE, DATA_DIR, RESULTS_DIR, CKPT_DIR, MODELS_DIR, TFLITE_DIR]:
    os.makedirs(d, exist_ok=True)

SESSION_START = time.time()
print(f"Drive base : {DRIVE_BASE}")
print("Setup complete ✓")

In [ ]:
%%capture install_out
!pip install -q tensorflow-model-optimization tf2onnx onnx scikit-learn pillow tqdm
import tensorflow_model_optimization as tfmot
import tf2onnx
print("All packages installed ✓")

In [ ]:
# ── Kaggle credentials (run once per session) ──────────────────────────────
import os
KAGGLE_JSON = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(KAGGLE_JSON):
    from google.colab import files
    print("Upload your kaggle.json API token (from kaggle.com → Settings → API):")
    files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials saved ✓")
else:
    print("Kaggle credentials already present ✓")

In [ ]:
# ── Download Kvasir-SEG ─────────────────────────────────────────────────────
KVASIR_DIR = f"{DATA_DIR}/kvasir"
os.makedirs(KVASIR_DIR, exist_ok=True)
if not any(os.scandir(KVASIR_DIR)):
    print("Downloading Kvasir-SEG...")
    # Try Kaggle dataset first
    !kaggle datasets download -d debeshjha1/kvasirseg -p {KVASIR_DIR} 2>/dev/null ||      kaggle datasets download -d gt1086/kvasir-seg -p {KVASIR_DIR} 2>/dev/null ||      (echo "Kaggle download failed, trying direct URL..." &&       wget -q https://datasets.simula.no/downloads/kvasir-seg.zip -O {KVASIR_DIR}/kvasir-seg.zip)
    !cd {KVASIR_DIR} && unzip -q "*.zip" && rm -f *.zip
    print("Download complete ✓")
else:
    print(f"Kvasir-SEG cached ✓")

# Find images and masks dirs
img_dir = mask_dir = None
for root,dirs,files in os.walk(KVASIR_DIR):
    for d in dirs:
        dl=d.lower()
        if dl in ("images","image") and img_dir is None: img_dir=os.path.join(root,d)
        if dl in ("masks","mask") and mask_dir is None: mask_dir=os.path.join(root,d)
print(f"Images: {img_dir}"); print(f"Masks:  {mask_dir}")
img_files = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg','.png'))])
print(f"Total samples: {len(img_files)}")

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
IMG_SIZE   = 256
BATCH_SIZE = 16
SEEDS      = [42, 123, 456, 789, 1024]
CFG = {
    "baseline":       {"epochs":50,"lr":1e-4,"patience":10},
    "qat":            {"epochs":15,"lr":1e-5},
    "student_scratch":{"epochs":50,"lr":1e-4,"patience":10},
    "kd":             {"epochs":50,"lr":1e-4,"patience":10,"temperature":3.0,"alpha":0.6},
}

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def compute_cls_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob.ravel() >= thr).astype(int)
    y_true = y_true.astype(int).ravel()
    auc  = roc_auc_score(y_true, y_prob.ravel())
    f1   = f1_score(y_true, y_pred, zero_division=0)
    sens = recall_score(y_true, y_pred, zero_division=0)
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return {"auc":float(auc),"f1":float(f1),"sensitivity":float(sens),"specificity":float(spec)}

def export_tflite(model, path, quant="fp32", calib_ds=None, n_calib=200):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quant == "int8":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if calib_ds:
            samples = []
            for imgs, _ in calib_ds:
                for i in range(len(imgs)):
                    samples.append(imgs[i].numpy())
                    if len(samples) >= n_calib: break
                if len(samples) >= n_calib: break
            def rep():
                for s in samples: yield [np.expand_dims(s,0)]
            conv.representative_dataset = rep
            conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            conv.inference_input_type  = tf.uint8
            conv.inference_output_type = tf.uint8
    elif quant == "fp16":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]
    data = conv.convert()
    with open(path,"wb") as f: f.write(data)
    mb = os.path.getsize(path)/1e6
    print(f"  TFLite {quant}: {os.path.basename(path)} ({mb:.1f} MB)")
    return mb

def eval_tflite(path, dataset, warmup=5):
    interp = tf.lite.Interpreter(model_path=path, num_threads=4)
    interp.allocate_tensors()
    inp_d = interp.get_input_details()[0]
    out_d = interp.get_output_details()[0]
    preds, labels, lats = [], [], []
    for imgs, lbls in dataset:
        for i in range(len(imgs)):
            x = np.expand_dims(imgs[i].numpy(), 0)
            if inp_d["dtype"] == np.uint8:
                sc,zp = inp_d["quantization"]; x=(x/sc+zp).astype(np.uint8)
            interp.set_tensor(inp_d["index"], x)
            t0=time.perf_counter(); interp.invoke(); t1=time.perf_counter()
            lats.append((t1-t0)*1000)
            r=interp.get_tensor(out_d["index"])
            if out_d["dtype"]==np.uint8:
                sc,zp=out_d["quantization"]; r=(r.astype(np.float32)-zp)*sc
            preds.append(r.squeeze()); labels.append(lbls[i].numpy())
    lats = np.array(lats[warmup:])
    m = compute_cls_metrics(np.array(labels), np.array(preds))
    m["latency_ms"] = float(np.median(lats))
    m["latency_p95_ms"] = float(np.percentile(lats,95))
    m["size_mb"] = os.path.getsize(path)/1e6
    print(f"  TFLite AUC={m['auc']:.4f}  lat={m['latency_ms']:.1f}ms  sz={m['size_mb']:.1f}MB")
    return m

def done_flag(name): return f"{RESULTS_DIR}/{name}.done"
def is_done(name): return os.path.exists(done_flag(name))
def mark_done(name, result):
    with open(f"{RESULTS_DIR}/{name}.json","w") as f: json.dump(result, f)
    open(done_flag(name),"w").close()
def load_result(name):
    with open(f"{RESULTS_DIR}/{name}.json") as f: return json.load(f)

def train_model(model, train_ds, val_ds, cfg, name, monitor="val_auc", mode="max"):
    """Train with Drive checkpoint + resume. Returns history."""
    ckpt = f"{CKPT_DIR}/{name}_best.keras"
    epoch_f = f"{CKPT_DIR}/{name}_epoch.json"
    initial_epoch = 0
    if os.path.exists(ckpt):
        try:
            model.load_weights(ckpt)
            if os.path.exists(epoch_f):
                initial_epoch = json.load(open(epoch_f)).get("epoch",0)
            print(f"  Resumed from epoch {initial_epoch} ✓")
        except Exception as e:
            print(f"  Checkpoint load failed ({e}), starting fresh")
    class _EpochLog(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            json.dump({"epoch":epoch+1}, open(epoch_f,"w"))
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt, save_best_only=True, monitor=monitor, mode=mode, verbose=0),
        tf.keras.callbacks.EarlyStopping(
            patience=cfg.get("patience",7), monitor=monitor, mode=mode,
            restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=3, mode=mode, verbose=0),
        _EpochLog()
    ]
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=cfg.get("epochs",20), initial_epoch=initial_epoch,
        class_weight=cfg.get("class_weight"), callbacks=cbs, verbose=1)
    return model, history

print("Utilities loaded ✓")

In [ ]:
# ── Data pipeline ───────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
train_files, test_files = train_test_split(img_files, test_size=0.15, random_state=42)
train_files, val_files  = train_test_split(train_files, test_size=0.15/0.85, random_state=42)
print(f"Splits: train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")

def make_ds(files, augment=False, shuffle=False):
    def load(fname):
        fname = fname.numpy().decode()
        base = os.path.splitext(fname)[0]
        img_path  = os.path.join(img_dir,  fname)
        mask_path = None
        for ext in ['.jpg','.png']:
            p = os.path.join(mask_dir, base+ext)
            if os.path.exists(p): mask_path=p; break
        img  = tf.image.decode_image(tf.io.read_file(img_path),  channels=3, expand_animations=False)
        mask = tf.image.decode_image(tf.io.read_file(mask_path), channels=1, expand_animations=False)
        img  = tf.image.resize(img,  [IMG_SIZE,IMG_SIZE])
        mask = tf.image.resize(mask, [IMG_SIZE,IMG_SIZE])
        img  = tf.cast(img, tf.float32)/127.5-1.0
        mask = tf.cast(mask>128, tf.float32)
        return img, mask
    def aug(img, mask):
        seed = tf.random.uniform(shape=(),minval=0,maxval=2**31-1,dtype=tf.int32)
        img  = tf.image.stateless_random_flip_left_right(img,  seed=(seed,seed))
        mask = tf.image.stateless_random_flip_left_right(mask, seed=(seed,seed))
        img  = tf.image.random_brightness(img, 0.15)
        return img, mask
    def py_load(fname): return tf.py_function(load,[fname],[tf.float32,tf.float32])
    ds = tf.data.Dataset.from_tensor_slices(files)
    if shuffle: ds=ds.shuffle(len(files),reshuffle_each_iteration=True)
    ds = ds.map(py_load, num_parallel_calls=tf.data.AUTOTUNE)
    if augment: ds=ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda i,m: (tf.ensure_shape(i,[IMG_SIZE,IMG_SIZE,3]),
                               tf.ensure_shape(m,[IMG_SIZE,IMG_SIZE,1])))
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_files, True, True)
val_ds   = make_ds(val_files)
test_ds  = make_ds(test_files)

In [ ]:
# ── Models ──────────────────────────────────────────────────────────────────
from tensorflow.keras import layers

def conv_block(x, filters, dropout=0.0):
    x = layers.Conv2D(filters,3,padding="same",use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters,3,padding="same",use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    if dropout>0: x=layers.SpatialDropout2D(dropout)(x)
    return x

def build_unet(filters=(64,128,256,512), name="unet"):
    inp = keras.Input((IMG_SIZE,IMG_SIZE,3))
    skips, x = [], inp
    for f in filters:
        x = conv_block(x,f,0.1); skips.append(x)
        x = layers.MaxPooling2D(2)(x)
    x = conv_block(x,filters[-1]*2,0.2)
    for f in reversed(filters):
        x = layers.Conv2DTranspose(f,2,2,padding="same")(x)
        x = layers.Concatenate()([x,skips.pop()])
        x = conv_block(x,f,0.1)
    return keras.Model(inp, layers.Conv2D(1,1,activation="sigmoid")(x), name=name)

def build_unet_lite(name="unet_lite"):
    return build_unet((32,64,128,256), name=name)

def dice_coef(y_true, y_pred, smooth=1e-7):
    y_true_f=tf.reshape(y_true,[-1]); y_pred_f=tf.reshape(y_pred,[-1])
    intersection=tf.reduce_sum(y_true_f*y_pred_f)
    return (2.*intersection+smooth)/(tf.reduce_sum(y_true_f)+tf.reduce_sum(y_pred_f)+smooth)
def dice_loss(y_true,y_pred): return 1-dice_coef(y_true,y_pred)
def bce_dice(y_true,y_pred): return keras.losses.binary_crossentropy(y_true,y_pred)+dice_loss(y_true,y_pred)

def eval_seg(model, ds, tag=""):
    dices,ious=[],[]
    for imgs,masks in ds:
        p=model(imgs,training=False).numpy(); m=masks.numpy()
        pb=(p>0.5).astype(np.float32); mb=m.astype(np.float32)
        inter=(pb*mb).sum((1,2,3)); union=pb.sum((1,2,3))+mb.sum((1,2,3))
        dices.extend((2*inter+1e-7)/(union+1e-7)); ious.extend((inter+1e-7)/(union-inter+1e-7))
    d,i=np.mean(dices),np.mean(ious)
    print(f"  [{tag}] Dice={d:.4f}  IoU={i:.4f}")
    return {"dice":float(d),"iou":float(i)}

print("U-Net models defined ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENTS (Baseline → QAT → Scratch → KD → KD+QAT)
# ══════════════════════════════════════════════════════════════════════════════
import tensorflow_model_optimization as tfmot
baseline_r, scratch_r, kd_r, qat_r = [], [], [], []

for seed in SEEDS:
    set_seed(seed)
    # Baseline
    key=f"baseline_s{seed}"
    if is_done(key): baseline_r.append(load_result(key)); print(f"⏩ {key}")
    else:
        m=build_unet()
        m.compile(keras.optimizers.Adam(CFG["baseline"]["lr"]), bce_dice,
                  metrics=[dice_coef,keras.metrics.MeanIoU(2,name="miou")])
        m,_=train_model(m,train_ds,val_ds,CFG["baseline"],key,"val_dice_coef")
        metrics=eval_seg(m,test_ds,key); m.save(f"{MODELS_DIR}/unet_s{seed}.keras")
        mark_done(key,metrics); baseline_r.append(metrics)
    # Scratch student
    key=f"scratch_s{seed}"
    if is_done(key): scratch_r.append(load_result(key)); print(f"⏩ {key}")
    else:
        s=build_unet_lite()
        s.compile(keras.optimizers.Adam(CFG["student_scratch"]["lr"]), bce_dice,
                  metrics=[dice_coef])
        s,_=train_model(s,train_ds,val_ds,CFG["student_scratch"],key,"val_dice_coef")
        metrics=eval_seg(s,test_ds,key); s.save(f"{MODELS_DIR}/unet_lite_scratch_s{seed}.keras")
        mark_done(key,metrics); scratch_r.append(metrics)

print(f"Baseline Dice: {np.mean([r['dice'] for r in baseline_r]):.4f}")
print(f"Scratch  Dice: {np.mean([r['dice'] for r in scratch_r]):.4f}")

In [ ]:
# ─ KD Segmentation ──────────────────────────────────────────────────────────
teacher = keras.models.load_model(f"{MODELS_DIR}/unet_s42.keras",
                                   custom_objects={"bce_dice":bce_dice,"dice_coef":dice_coef})
T=CFG["kd"]["temperature"]; ALPHA=CFG["kd"]["alpha"]

for seed in SEEDS:
    key=f"kd_s{seed}"
    if is_done(key): kd_r.append(load_result(key)); print(f"⏩ {key}"); continue
    set_seed(seed)
    student=build_unet_lite(); opt=keras.optimizers.Adam(CFG["kd"]["lr"])
    best_dice,patience_c=0,0; ckpt=f"{CKPT_DIR}/kd_s{seed}.keras"
    if os.path.exists(ckpt): student.load_weights(ckpt); print("  Resumed ✓")
    for epoch in range(CFG["kd"]["epochs"]):
        for imgs,masks in train_ds:
            with tf.GradientTape() as tape:
                t_out=teacher(imgs,training=False); s_out=student(imgs,training=True)
                # Pixel-level soft targets
                ts=tf.nn.sigmoid((t_out*2-1)/T); ss=tf.nn.sigmoid((s_out*2-1)/T)
                kl=ts*tf.math.log((ts+1e-7)/(ss+1e-7))+(1-ts)*tf.math.log((1-ts+1e-7)/(1-ss+1e-7))
                loss=ALPHA*tf.reduce_mean(kl)*(T**2)+(1-ALPHA)*bce_dice(masks,s_out)
            grads=tape.gradient(loss,student.trainable_variables)
            opt.apply_gradients(zip(grads,student.trainable_variables))
        val_dice=eval_seg(student,val_ds,f"ep{epoch+1}")["dice"]
        if val_dice>best_dice: best_dice=val_dice; patience_c=0; student.save_weights(ckpt)
        else:
            patience_c+=1
            if patience_c>=CFG["kd"]["patience"]: break
    student.load_weights(ckpt)
    metrics=eval_seg(student,test_ds,key)
    student.save(f"{MODELS_DIR}/unet_lite_kd_s{seed}.keras"); mark_done(key,metrics); kd_r.append(metrics)

# QAT on best KD model
for seed in SEEDS:
    key=f"qat_kd_s{seed}"
    if is_done(key): qat_r.append(load_result(key)); print(f"⏩ {key}"); continue
    base=keras.models.load_model(f"{MODELS_DIR}/unet_lite_kd_s{seed}.keras",
                                  custom_objects={"bce_dice":bce_dice,"dice_coef":dice_coef})
    qat_m=tfmot.quantization.keras.quantize_model(base)
    qat_m.compile(keras.optimizers.Adam(1e-5),bce_dice,metrics=[dice_coef])
    qat_m.fit(train_ds,validation_data=val_ds,epochs=15,verbose=1)
    stripped=tfmot.quantization.keras.strip_pruning(qat_m)
    p=f"{TFLITE_DIR}/unet_lite_kd_qat_int8_s{seed}.tflite"
    conv=tf.lite.TFLiteConverter.from_keras_model(stripped)
    conv.optimizations=[tf.lite.Optimize.DEFAULT]
    with open(p,"wb") as f: f.write(conv.convert())
    metrics=eval_seg(stripped,test_ds,key); metrics["size_mb"]=os.path.getsize(p)/1e6
    mark_done(key,metrics); qat_r.append(metrics)

print(f"\nKD Dice:     {np.mean([r['dice'] for r in kd_r]):.4f}")
print(f"KD+QAT Dice: {np.mean([r.get('dice',0) for r in qat_r]):.4f}")
table=pd.DataFrame([{"method":"Baseline","dice":np.mean([r["dice"] for r in baseline_r])},
                     {"method":"Scratch", "dice":np.mean([r["dice"] for r in scratch_r])},
                     {"method":"KD",      "dice":np.mean([r["dice"] for r in kd_r])},
                     {"method":"KD+QAT",  "dice":np.mean([r.get("dice",0) for r in qat_r])}])
table.to_csv(f"{RESULTS_DIR}/kvasir_results.csv",index=False)
print(table.to_string(index=False)); print(f"\nSaved: {RESULTS_DIR}")